# HydraY NNUE — 8 king bucket, riaperto al budget giusto

Runtime → **GPU (T4)**, poi "Esegui tutte". ~7h in **sedici tappe**, quindi
quasi certamente su piu' sessioni: ogni tappa lascia un checkpoint verificato su
Drive e si riparte da li'.

### Perche' riaprire una domanda gia' chiusa

`A4-bis CHIUSA DEFINITIVAMENTE 2026-08-01: 4 bucket.` Due SPRT concordi,
**-12,37 +-9,34** e **-10,43 +-8,41**, su 1,18B di posizioni a **40 superbatch**.

Sei giorni dopo abbiamo misurato che **80 superbatch contro 40 valgono +29,4**,
e per quel motivo abbiamo ritirato A5 con questa motivazione: *"misurava a 40
SB, budget 4x troppo piccolo, quindi confrontava due reti entrambe
sotto-addestrate"*.

Gli 8 bucket hanno **esattamente il doppio dei parametri di l0** rispetto ai 4
(payload verificato: 6.308.880 B contro 3.163.152 B nell'era a 512 neuroni,
fattore 2 esatto). A budget uguale ricevono meta' addestramento per parametro.
E' lo stesso identico argomento che la memoria applica gia' a HalfKA contro 768:

> HalfKA ha x4 i parametri di l0, quindi era ~1/16 di addestramento per
> parametro: **perdere era l'esito atteso** e il risultato non dice nulla
> sull'architettura.

Non l'abbiamo riconosciuto quando la stessa cosa e' successa a 8 contro 4.

### C'e' anche un argomento di merito, non solo di budget

La mappa a 4 bucket mette **le traverse 3-8 tutte nello stesso bucket**: un re
su traversa 7 e uno su traversa 3 condividono i pesi. I nostri sanity dicono che
i re attivi erano un punto debole misurato (23 -> 116 con v7). La mappa a 8
separa proprio quello: traverse 3-4 -> bucket 6, traverse 5-8 -> bucket 7, e
alza la risoluzione sulla prima traversa (per colonna invece che a coppie).

### Il disegno dell'esperimento

Stesso dataset (v7, 2,97B), stesso budget del campione (320 SB), stesso
binario. **L'unica variabile e' la mappa dei king bucket.**

I checkpoint sono salvati ogni 20 superbatch: quello a **160** serve, non
cancellarlo. Con due punti (160 e 320) si legge la *pendenza* del budget, ed e'
quella che decide se 8 bucket sono peggiori davvero o solo ancora affamati:

- 8@320 **batte** 4@320 -> la mappa e' migliore, si adotta
- 8@320 perde ma la sua pendenza 160->320 e' molto piu' ripida del +2,7 dei
  4 bucket -> sta ancora salendo, e allora vale spendere i 640
- 8@320 perde **e** la pendenza e' piatta come quella dei 4 -> sature
  entrambe, gli 8 bucket sono genuinamente peggiori: chiusa per sempre


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + configurazione ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]

PARTS = {i: find(f'hydray_v7_part{i}.bin.zst') for i in (1, 2, 3, 4)}
# Taglia attesa DOPO la decompressione, per parte. Una decompressione
# interrotta a meta' produce un file piu' corto e nessun errore: senza questo
# controllo si addestrerebbe in silenzio su dati troncati.
RAW_SIZE = {1: 23_742_906_368, 2: 23_742_906_368,
            3: 23_742_906_368, 4: 23_745_983_264}
for i, p in PARTS.items():
    print(f'parte {i}: {os.path.getsize(p)/2**30:6.2f} GiB compressa  {p}')

NET_ID   = 'hydray-h8-v7-320sb'
TOTAL_SB = 320          # stesso budget del campione: l'unica variabile e' la mappa
STAGE    = 20           # sedici tappe
ORDER    = [1, 2, 3, 4] * 4           # ogni fetta girata quattro volte
TRAINER  = '/content/th/nnue/trainer'
assert len(ORDER) * STAGE == TOTAL_SB and STAGE % 10 == 0

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
BRANCH = 'halfka8-v2'
sh('rm -rf /content/th')
sh(f'git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY /content/th')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 1024;' in src, 'NON e il branch a 1024 neuroni'
assert 'const INPUT_BUCKETS: usize = 8;' in src, 'NON e il branch a 8 king bucket'
tr = open(f'{TRAINER}/src/bin/trainer.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer.rs non e a 1024'
# La mappa e' il punto dell'esperimento: verificarla, non fidarsi del nome del branch.
import re as _re
lay = _re.search(r'const BUCKET_LAYOUT: \[usize; 32\] = \[(.*?)\];', tr, _re.S).group(1)
lay = [int(x) for x in _re.findall(r'\d+', lay)]
assert sorted(set(lay)) == list(range(8)), f'la mappa non usa 8 bucket: {sorted(set(lay))}'
assert lay == [0,1,2,3, 4,4,5,5, 6,6,6,6, 6,6,6,6, 7,7,7,7, 7,7,7,7, 7,7,7,7, 7,7,7,7], lay
print(f'branch {BRANCH}, 1024 neuroni, 8 king bucket, mappa verificata: ok')

sh('apt-get -qq install -y zstd >/dev/null')
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~36 GiB (una fetta + cache di Drive)')
assert free_gb > 45, 'disco insufficiente'

In [ ]:
# --- helper delle tappe (ESEGUIRE SEMPRE, anche in ripartenza) ---

def load_slice(n):
    """Scompatta la fetta n in /content/data.bin, sostituendo la precedente."""
    if os.path.exists('/content/data.bin'):
        os.remove('/content/data.bin')          # spazio prima, non dopo
    sh(f'zstd -d -T0 --long=27 -c "{PARTS[n]}" > /content/data.bin')
    got = os.path.getsize('/content/data.bin')
    assert got == RAW_SIZE[n], f'fetta {n} troncata: {got} != {RAW_SIZE[n]}'
    print(f'fetta {n}: {got//32/1e6:.1f}M posizioni, taglia verificata', flush=True)

def stage_cmd(end, start, resume_from):
    """⚠️ STAGE_END DEVE STARE ATTACCATO A `cargo`, non in testa alla riga.
    `STAGE_END=40 cd dir && cargo ...` assegna la variabile SOLO a `cd`: cargo
    la riceve vuota, il trainer ignora le tappe e tira dritto fino a TOTAL_SB
    senza salvare niente. E' costato un run intero. Da qui l'`env` esplicito."""
    args = f'/content/data.bin {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer --features cuda -- {args}')

def save_to_drive(end):
    """Copia il checkpoint su Drive e VERIFICA che ci sia arrivato davvero: il
    mount di Drive scrive attraverso una cache, quindi un upload mai completato
    passerebbe per riuscito."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'/content/drive/MyDrive/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint mancante in locale: {ck}'
    sh(f'rm -rf {dst} && cp -r {ck} /content/drive/MyDrive/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f))
                         for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst), f'la copia su Drive non esiste: {dst}'
    assert size(dst) == size(ck), f'copia su Drive incompleta: {size(dst)} != {size(ck)}'
    print(f'tappa fino al superbatch {end} su Drive ({size(dst)/2**20:.0f} MiB, verificata)', flush=True)

def run_stages(done=0):
    """Esegue le tappe da `done` in poi. done=0 parte da zero."""
    assert done % STAGE == 0, f'{done} non e un confine di tappa'
    prev = done if done else None
    for k in range(done // STAGE, len(ORDER)):
        end, start, sl = (k+1)*STAGE, k*STAGE + 1, ORDER[k]
        print(f'\n===== tappa {k+1}/{len(ORDER)}: superbatch {start}-{end}, fetta {sl} =====', flush=True)
        load_slice(sl)
        sh(stage_cmd(end, start, prev))
        save_to_drive(end)
        prev = end

In [ ]:
# --- training: otto tappe, fette 1-2-3-4-1-2-3-4 ---
# NON eseguire questa cella in una ripartenza: usa invece la cella in fondo.
run_stages(done=0)

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
assert 12617744 <= sz < 12617744 + 64, f'taglia {sz}: NON e la rete a 8 bucket/1024'
print('quantised.bin:', sz, 'byte — 8 king bucket x 1024 neuroni confermati\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*70)
print('RIFERIMENTO — la rete adottata (v7, 2,97B, 4 bucket), in locale:')
print('  startpos            48      mediogioco ~24 pezzi   930')
print('  KQvK               929      KRPvKR                 110')
print('  cavallo in piu     730      re attivi (finale)     116')
print('  donna in piu      1824      training loss      0.012734')
print()
print('La training loss E confrontabile: stesso dataset e stesso budget, cambia')
print('solo la mappa. Ma ATTENZIONE — una loss migliore NON basta: e gia')
print('successo (rete a 8 bucket, 2026-08-01) di avere train loss migliore, NPS')
print('pari e gioco peggiore. Guarda soprattutto i RE ATTIVI: e la casella che')
print('la mappa a 4 bucket non sa distinguere. Il verdetto e comunque lo SPRT.')
print()
print('NON CANCELLARE il checkpoint a 160 superbatch su Drive: serve per')
print('leggere la pendenza del budget (vedi la cella di testa).')
print('='*70)

In [ ]:
# --- RIPARTENZA (usare SOLO se la sessione e' morta a meta') ---
# Come si usa:
#   1. esegui le celle da "helper" fino a "helper delle tappe" compresa;
#   2. NON eseguire la cella del training;
#   3. metti RESUME = True e DONE = ultimo superbatch salvato su Drive.
# La fetta giusta viene ricavata da ORDER: non devi ricordarti dov'era.
#
# Con RESUME = False questa cella non fa niente, cosi' "Esegui tutte" e' sicuro
# (altrimenti, a run finito, ripartirebbe da DONE rifacendo ore di training).

RESUME = False
DONE   = 20      # ultimo superbatch salvato su Drive

if not RESUME:
    print('ripartenza disattivata (RESUME = False) — nessuna azione')
else:
    ck = f'/content/drive/MyDrive/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint non trovato su Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r {ck} {TRAINER}/checkpoints/')
    assert os.path.isdir(f'{TRAINER}/checkpoints/{NET_ID}-{DONE}')
    print(f'ripartenza dal superbatch {DONE} (prossima fetta: {ORDER[DONE//STAGE]})\n')
    run_stages(done=DONE)

## Come leggere il risultato

Lo SPRT e' **testa a testa** contro la rete adottata. Attenzione: qui, a
differenza degli esperimenti sui dati, i due lati NON possono condividere il
binario — la mappa dei king bucket e' compilata dentro il motore. Servono due
build, e va verificato che l'unica differenza sia quella.

**Se vince** — la mappa a 4 bucket stava buttando via risoluzione, e la
decisione del 2026-08-01 era un artefatto del budget, esattamente come A5. Si
adotta e si guarda subito se a 640 superbatch sale ancora.

**Se pareggia** — a parita' di budget la risoluzione in piu' non paga. NON
chiude la domanda: con il doppio dei parametri, pareggiare a parita' di budget
significa che a parametro sta imparando meglio. Guarda la pendenza 160->320.

**Se perde con pendenza ripida** — sta ancora salendo, il budget non basta
ancora. Il ramo giusto sono i 640 superbatch, non l'abbandono.

**Se perde con pendenza piatta** — sature entrambe, a parita' di budget e di
dati. Allora gli 8 bucket sono genuinamente peggiori e la domanda si chiude per
davvero, con l'argomento che nel 2026-08-01 non avevamo.

### Il costo che non e' nell'Elo

Il file della rete raddoppia: 12,6 MB contro 6,3. Finisce dentro il binario e
dentro la cache. Il percorso di refresh dell'accumulatore tocca tabelle il
doppio piu' grandi, quindi **va misurato l'NPS**, non dato per invariato: se la
rete vince di poco ma costa NPS, il conto va rifatto.
